<a href="https://colab.research.google.com/github/sararahman1729/Custom-Loss-Function/blob/CUB200/cub200.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.models import resnet50, densenet121
import numpy as np
import copy
from sklearn.metrics import confusion_matrix

# Define custom loss with entropy regularization
class CrossEntropyWithEntropyRegularization(nn.Module):
    def __init__(self, alpha=0.5, beta=0.1):
        super(CrossEntropyWithEntropyRegularization, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, outputs, targets):
        # Cross-Entropy Loss
        ce_loss = nn.CrossEntropyLoss()(outputs, targets)
        # Entropy Regularizer
        entropy_loss = -torch.sum(torch.softmax(outputs, dim=1) * torch.log_softmax(outputs, dim=1), dim=1).mean()
        # Total Loss
        total_loss = (1 - self.alpha) * ce_loss + self.beta * entropy_loss
        return total_loss

# Defining training function
def train_model(model, dataloaders, criterion, optimizer, num_epochs=25, device='cuda'):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    best_hyperparams = {'alpha': criterion.alpha, 'beta': criterion.beta}

    for epoch in range(num_epochs):
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                best_hyperparams = {'alpha': criterion.alpha, 'beta': criterion.beta}

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model, best_acc, best_hyperparams

# Dataset and DataLoader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load datasets and create random split for train/validation
dataset = datasets.ImageFolder('path_to_data', transform=transform)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

dataloaders = {
    'train': DataLoader(train_dataset, batch_size=32, shuffle=True),
    'val': DataLoader(val_dataset, batch_size=32, shuffle=True)
}

# Initialization of models, criterion, optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = resnet50(pretrained=True).to(device)

# Dynamic parameter adjustment for alpha, beta
for alpha in np.linspace(0.1, 0.9, 5):
    for beta in np.linspace(0.1, 0.5, 5):
        criterion = CrossEntropyWithEntropyRegularization(alpha=alpha, beta=beta)
        optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

        print(f'\nTraining with alpha={alpha}, beta={beta}')
        model, best_acc, best_hyperparams = train_model(model, dataloaders, criterion, optimizer, num_epochs=150 if isinstance(model, resnet50) else 200, device=device)
        print(f'Best validation accuracy: {best_acc:.4f} with alpha={best_hyperparams["alpha"]}, beta={best_hyperparams["beta"]}')

#best model and confusion matrix
torch.save(model.state_dict(), 'best_model.pth')
